In [3]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

In [4]:
#load trained model,pickle file,scaler,onehotencoder
model=load_model("model.h5")
with open("scaler.pkl","rb") as file:
    scaler=pickle.load(file)
with open("one_hot_encoder_geo.pkl","rb") as file:
    one_hot_encoder_geo=pickle.load(file)

with open("label_encoder.pkl","rb") as file:
    label_encoder=pickle.load(file)

In [5]:
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [9]:
#one-hot encode to geography
geo_encoded=one_hot_encoder_geo.transform([[input_data['Geography']]]).toarray()
geo_encoded_df=pd.DataFrame(geo_encoded,columns=one_hot_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

c:\Users\hband\anaconda3\envs\tfenv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [10]:
input_df=pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [13]:
#encode categorical feature gender
input_df["Gender"] = label_encoder.transform(input_df["Gender"])
input_df


,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [14]:
#concatenate one-hot encoded geography with input_df
input_df=pd.concat([input_df.drop("Geography",axis=1),geo_encoded_df],axis=1)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [15]:
#scaling
input_scaled=scaler.transform(input_df)
input_scaled

array([[-0.47154541,  0.90911166,  0.09477172, -0.69844549, -0.29010416,
         0.80510537,  0.63367318,  0.95214374, -0.84805047,  0.98019606,
        -0.57581067, -0.56349184]])

In [20]:
#predict
prediction=model.predict(input_scaled)

1/1 [==============================] - 0s 20ms/step


In [21]:
prediction_prob=prediction[0][0]
if prediction_prob>=0.5:
    print(f"customer is likely to churn with a probablity of {prediction_prob:.2f}")
else:    print(f"customer is not likely to churn with a probablity of {prediction_prob:.2f}")

customer is not likely to churn with a probablity of 0.03
